In [1]:
from utils.load import load_json_or_jsonl
import pandas as pd

from pathlib import Path
INPUT  = Path("../../data/dataset/applets/applets_real_new.jsonl")
OUT = Path("../../data/dataset/applets/applets_real_new_bis.jsonl")

In [2]:
TRIGGER = Path("../../data/ifttt_catalog/triggers.json")
ACTION  =Path("../../data/ifttt_catalog/actions.json")
applets = load_json_or_jsonl(INPUT)
triggers = load_json_or_jsonl(TRIGGER)
actions  = load_json_or_jsonl(ACTION)
trigger_index = {t["api_endpoint_slug"]: t for t in triggers}
action_index = {a["api_endpoint_slug"]: a for a in actions}
df_orig = pd.DataFrame(applets)

In [3]:
df_orig

,by_service_owner,channels,description,friendly_id,id,installs_count,name,pro_features,requires_android_app,requires_ios_app,...,actions_category,filter_code,trigger_apis,action_apis,row_index,rule_description,user_intent_example,instruct_zs,instruct_ft,combined_len
0,True,"[{'name': 'Chicago Transit Authority', 'module...",Commuting in Chicago? This Applet posts to a S...,CrxshJV6-post-to-slack-when-delays-affect-your...,CrxshJV6,3.0,Post to Slack when delays affect your morning ...,True,False,False,...,['Communication'],var Hour = Meta.currentUserTime.hour()\r\nvar ...,[cta.new_pink_line_alert],[slack.post_to_channel],0,This automation posts a message to a designate...,I'd like to receive real-time updates in my Sl...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,818
1,True,"[{'name': 'Notifications', 'module_name': 'if_...",This Applet will send you a notification when ...,UMq6ryuD-get-notified-when-a-nj-transit-adviso...,UMq6ryuD,34.0,Get notified when a NJ transit advisory affect...,True,False,False,...,['Notifications'],var Hour = Meta.currentUserTime.hour()\r\nvar ...,[nj_transit.new_bus_advisory],[if_notifications.send_notification],1,This automation sends you a high-priority noti...,Please notify me if there’s a new NJ transit b...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,742
2,True,"[{'name': 'Notifications', 'module_name': 'if_...",This Applet helps you work around commute disr...,K9GSUniy-get-notified-when-there-s-a-rider-ale...,K9GSUniy,7.0,Get notified when there's a rider alert on Dal...,True,False,False,...,['Notifications'],var Hour = Meta.currentUserTime.hour()\r\nvar ...,[dart.new_dart_rider_alert],[if_notifications.send_notification],2,This automation sends a high-priority notifica...,Please notify me immediately if there's a serv...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,821
3,False,"[{'name': 'Twitter', 'module_name': 'twitter',...",Get notified when @Wario64 (usually the fastes...,HaLPjmCQ-super-nintendo-classic-in-stock-alerts,HaLPjmCQ,377.0,Super Nintendo Classic in stock alerts,True,False,False,...,['Notifications'],"if (Twitter.newTweetByUser.Text.indexOf(""SNES""...",[twitter.new_tweet_by_user],[if_notifications.send_notification],3,This automation sends you a high-priority noti...,Please notify me immediately on my phone whene...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,768
4,False,"[{'name': 'Twitter', 'module_name': 'twitter',...",Post Mastodon's Toot on Twitter(Exclude Mentions),K6FLEawj-mastodon-twitter-exclude-mentions,K6FLEawj,19.0,Mastodon→Twitter(Exclude Mentions),True,False,False,...,['Social networks'],"if(Feed.newFeedItem.EntryContent.indexOf(""@"") ...",[feed.new_feed_item],[twitter.post_new_tweet],4,This automation monitors a specified Mastodon ...,I'd like to automatically tweet new posts from...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,786
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,False,"[{'name': 'Twitter', 'module_name': 'twitter',...",SantaAdrianaV,wdgHbhsc-santaadrianav,wdgHbhsc,5.0,SantaAdrianaV,True,False,False,...,['Social networks'],//var timeOfDay = Meta.triggerTime.weekday()\r...,[twitter.new_tweet_by_user],[twitter.post_new_tweet],359,"If a specific Twitter user tweets, this automa...",I'd like to automatically share any tweets fro...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,1530
347,False,"[{'name': 'Twitter', 'module_name': 'twitter',...",RochelYaracuy,QPBbDg6v-rochelyaracuy,QPBbDg6v,5.0,RochelYaracuy,True,False,False,...,['Social networks'],//var timeOfDay = Meta.triggerTime.weekday()\r...,[twitter.new_tweet_by_user],[twitter.post_new_tweet],360,This automation detects every t

In [4]:
from llm_utility.requests.parallel.response import process_rows_D, MAX_WORKERS

res = process_rows_D(applets[:2],trigger_index,action_index, max_workers=MAX_WORKERS)

df_res = pd.DataFrame(res)
df_res.drop(columns=['app_summary'],inplace=True)


KeyboardInterrupt: 

In [12]:
df_plus=df_orig.merge(df_res,on='row_index', how='inner')
df_plus.dropna(subset=['user_intent_example'],inplace=True)
df_plus

NameError: name 'df_res' is not defined

In [5]:
import math
# --- funzione per pulire NaN ---
def replace_nan_with_none(obj):
    if isinstance(obj, float) and math.isnan(obj):
        return None
    if isinstance(obj, dict):
        return {k: replace_nan_with_none(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [replace_nan_with_none(x) for x in obj]
    return obj

In [15]:
import json

records = [replace_nan_with_none(r) for r in df_orig.to_dict(orient="records")]
for r in records:
    r['user_intent_example']=r['user_intent_example'].strip("\"")
    r['user_intent_example']=r['user_intent_example'].strip("'")

from llm_utility.prompts.prompt_in import make_prompt, get_prompt_config

PROMPT_CFG_ZS = get_prompt_config('catalog_full')
PROMPT_CFG_FT = get_prompt_config('minimal')

for r in records:
    r['instruct_zs']=make_prompt(r,trigger_index,action_index,PROMPT_CFG_ZS)
    r['instruct_ft']=make_prompt(r,trigger_index,action_index,PROMPT_CFG_FT)

df_update = pd.DataFrame(records)
df_update
import pandas as pd

df = pd.DataFrame(records)
df.to_json(OUT, orient="records", lines=True, force_ascii=False)


In [7]:
df

,by_service_owner,channels,description,friendly_id,id,installs_count,name,pro_features,requires_android_app,requires_ios_app,...,actions_category,filter_code,trigger_apis,action_apis,row_index,rule_description,user_intent_example,instruct_zs,instruct_ft,combined_len
0,True,"[{'name': 'Chicago Transit Authority', 'module...",Commuting in Chicago? This Applet posts to a S...,CrxshJV6-post-to-slack-when-delays-affect-your...,CrxshJV6,3.0,Post to Slack when delays affect your morning ...,True,False,False,...,['Communication'],var Hour = Meta.currentUserTime.hour()\r\nvar ...,[cta.new_pink_line_alert],[slack.post_to_channel],0,This automation posts a message to a designate...,I'd like to receive real-time updates in my Sl...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,818
1,True,"[{'name': 'Notifications', 'module_name': 'if_...",This Applet will send you a notification when ...,UMq6ryuD-get-notified-when-a-nj-transit-adviso...,UMq6ryuD,34.0,Get notified when a NJ transit advisory affect...,True,False,False,...,['Notifications'],var Hour = Meta.currentUserTime.hour()\r\nvar ...,[nj_transit.new_bus_advisory],[if_notifications.send_notification],1,This automation sends you a high-priority noti...,Please notify me if there’s a new NJ transit b...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,742
2,True,"[{'name': 'Notifications', 'module_name': 'if_...",This Applet helps you work around commute disr...,K9GSUniy-get-notified-when-there-s-a-rider-ale...,K9GSUniy,7.0,Get notified when there's a rider alert on Dal...,True,False,False,...,['Notifications'],var Hour = Meta.currentUserTime.hour()\r\nvar ...,[dart.new_dart_rider_alert],[if_notifications.send_notification],2,This automation sends a high-priority notifica...,Please notify me immediately if there's a serv...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,821
3,False,"[{'name': 'Twitter', 'module_name': 'twitter',...",Get notified when @Wario64 (usually the fastes...,HaLPjmCQ-super-nintendo-classic-in-stock-alerts,HaLPjmCQ,377.0,Super Nintendo Classic in stock alerts,True,False,False,...,['Notifications'],"if (Twitter.newTweetByUser.Text.indexOf(""SNES""...",[twitter.new_tweet_by_user],[if_notifications.send_notification],3,This automation sends you a high-priority noti...,Please notify me immediately on my phone whene...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,768
4,False,"[{'name': 'Twitter', 'module_name': 'twitter',...",Post Mastodon's Toot on Twitter(Exclude Mentions),K6FLEawj-mastodon-twitter-exclude-mentions,K6FLEawj,19.0,Mastodon→Twitter(Exclude Mentions),True,False,False,...,['Social networks'],"if(Feed.newFeedItem.EntryContent.indexOf(""@"") ...",[feed.new_feed_item],[twitter.post_new_tweet],4,This automation monitors a specified Mastodon ...,I'd like to automatically tweet new posts from...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,786
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,False,"[{'name': 'Twitter', 'module_name': 'twitter',...",SantaAdrianaV,wdgHbhsc-santaadrianav,wdgHbhsc,5.0,SantaAdrianaV,True,False,False,...,['Social networks'],//var timeOfDay = Meta.triggerTime.weekday()\r...,[twitter.new_tweet_by_user],[twitter.post_new_tweet],359,"If a specific Twitter user tweets, this automa...",I'd like to automatically share any tweets fro...,You are an expert JavaScript developer with sp...,You are an expert JavaScript developer with sp...,1530
347,False,"[{'name': 'Twitter', 'module_name': 'twitter',...",RochelYaracuy,QPBbDg6v-rochelyaracuy,QPBbDg6v,5.0,RochelYaracuy,True,False,False,...,['Social networks'],//var timeOfDay = Meta.triggerTime.weekday()\r...,[twitter.new_tweet_by_user],[twitter.post_new_tweet],360,This automation detects every t